# Import

In [93]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [94]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datasets import Dataset, DatasetDict
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoModel,
    AutoTokenizer,
    PreTrainedModel,
    Trainer,
    TrainingArguments,
)
import seaborn as sns
import matplotlib.pyplot as plt
import wandb
from types import SimpleNamespace
from dataclasses import dataclass
from typing import Any, Dict, List
from transformers import PreTrainedTokenizerBase

# Settings

In [95]:
PERSONAL_PATH = "/content/drive/MyDrive/AIRO/MNLP/homework 2/Task 1 - model 2"
FOLDER_PATH   = os.path.join(PERSONAL_PATH, "data")
OUTPUT_PATH   = os.path.join(PERSONAL_PATH, "models")
OUTPUT_FILES  = os.path.join(PERSONAL_PATH, "output")
TRAIN_PATH    = os.path.join(FOLDER_PATH, "manzoni_train_tokens.csv")
DEV_PATH      = os.path.join(FOLDER_PATH, "manzoni_dev_tokens.csv")
TEST_CSV      = os.path.join(FOLDER_PATH, "OOD_test.csv")
csv_path      = os.path.join(OUTPUT_FILES,"test_predictions.csv")

os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(OUTPUT_FILES, exist_ok=True)

In [96]:
SEQ_LEN     = 64
BATCH_SIZE  = 16
EPOCHS      = 20
LR          = 2e-5
FREEZE_N    = 3
DROPOUT     = 0.2
FOCAL_GAMMA = 2.0
TARGET_EOS_RATIO = 0.25

MODEL_NAME  = "Musixmatch/umberto-commoncrawl-cased-v1"
RUN_NAME    = "it-umberto-focal-2"
PROJECT     = "sentence-splitting_3"

In [97]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [98]:
set_seed(1)

In [99]:
PUNCT = {".", "?", "!", ":", ";", "…", "—", "–", "«", "»", "\u201d", "\u2019"}

# Utils

In [100]:
def read_tokens(path: str):
    df = pd.read_csv(path)
    return list(df.token.astype(str)), list(df.label.astype(int))

In [101]:
def chunkify(tokens: List[str], labels: List[int], size: int):
    """Split flat sequence into fixed‑length chunks; pad last chunk."""
    pad_tok, pad_lab = "[PAD]", -100
    chunks_t, chunks_l = [], []
    for start in range(0, len(tokens), size):
        t = tokens[start: start + size]
        l = labels[start: start + size]
        if len(t) < size:
            pad_len = size - len(t)
            t += [pad_tok] * pad_len
            l += [pad_lab] * pad_len
        chunks_t.append(t)
        chunks_l.append(l)
    return chunks_t, chunks_l

In [102]:
def smart_oversample(ds: Dataset, target_ratio: float) -> Dataset:
    """Duplicate chunks that contain ≥1 EOS until positive‑class ratio ≈ target."""
    has_eos = [int(1 in lab) for lab in ds["labels"]]
    pos_idx = [i for i, x in enumerate(has_eos) if x]
    n_pos, n_tot = len(pos_idx), len(has_eos)
    cur_ratio = n_pos / n_tot
    if cur_ratio >= target_ratio:
        return ds

    desired_pos = int(target_ratio * n_tot / (1 - target_ratio))
    extra_needed = desired_pos - n_pos
    dup_idx = random.choices(pos_idx, k=extra_needed)
    extra   = ds.select(dup_idx)
    return Dataset.from_dict({k: ds[k] + extra[k] for k in ds.column_names})

In [103]:
def compute_class_weights(labels_2d):
    flat = np.concatenate(labels_2d)
    mask = flat != -100
    flat = flat[mask]
    return compute_class_weight("balanced", classes=np.array([0, 1]), y=flat)

In [104]:
def make_features(tokens: List[str]):
    feats = []
    for i, tok in enumerate(tokens):
        is_punct = int(tok in PUNCT)
        next_cap = 0
        if i + 1 < len(tokens):
            nxt = tokens[i + 1]
            next_cap = int(nxt and nxt[0].isupper())
        feats.append([is_punct, next_cap])
    return feats

In [105]:
def tokenize_and_align(batch):
    enc = tokenizer(
        batch["tokens"],
        is_split_into_words=True,
        padding=False,
        truncation=False
    )
    aligned_labels, aligned_feats = [], []
    for i, word_ids in enumerate(enc.word_ids(batch_index=i) for i in range(len(batch["tokens"]))):
        labs  = batch["labels"][i]
        feats = make_features(batch["tokens"][i])
        tmp_lbl, tmp_feat, prev = [], [], None
        for w in word_ids:
            if w is None:
                tmp_lbl.append(-100)
                tmp_feat.append([0, 0])
            elif w != prev:
                tmp_lbl.append(labs[w])
                tmp_feat.append(feats[w])
                prev = w
            else:
                tmp_lbl.append(-100)
                tmp_feat.append([0, 0])
        aligned_labels.append(tmp_lbl)
        aligned_feats.append(tmp_feat)
    enc["labels"]   = aligned_labels
    enc["features"] = aligned_feats
    return enc

In [106]:
def compute_metrics(p):
    preds  = p.predictions.argmax(-1).ravel()
    labels = p.label_ids.ravel()
    mask   = labels != -100
    preds, labels = preds[mask], labels[mask]
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, labels=[1], average="binary")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision_EOS": prec, "recall_EOS": rec, "f1_EOS": f1}

In [107]:
def load_dataset():
    tr_tok, tr_lab = read_tokens(TRAIN_PATH)
    dv_tok, dv_lab = read_tokens(DEV_PATH)
    tr_t, tr_l = chunkify(tr_tok, tr_lab, SEQ_LEN)
    dv_t, dv_l = chunkify(dv_tok, dv_lab, SEQ_LEN)
    ds = DatasetDict({
        "train": Dataset.from_dict({"tokens": tr_t, "labels": tr_l}),
        "dev":   Dataset.from_dict({"tokens": dv_t, "labels": dv_l}),
    })
    ds["train"] = smart_oversample(ds["train"], TARGET_EOS_RATIO)
    return ds


In [108]:
def split_with_model(
    tokens: List[str],
    model: PreTrainedModel,
    tok: AutoTokenizer,
    seq_len: int = SEQ_LEN,
) -> List[str]:

    device = next(model.parameters()).device
    model.eval()

    sentences, pending, idx = [], [], 0
    with torch.no_grad():
        while idx < len(tokens):
            window = tokens[idx : idx + seq_len]
            enc = tok(
                window,
                is_split_into_words=True,
                truncation=True,
                max_length=seq_len,
                padding="max_length",
                return_tensors="pt",
            )
            word_ids = enc.word_ids(batch_index=0)
            enc = {k: v.to(device) for k, v in enc.items()}

            feats_raw = make_features(window)
            feats_aligned = []
            for w in word_ids:
                if w is None:
                    feats_aligned.append([0, 0])
                else:
                    feats_aligned.append(feats_raw[w])
            feat_tensor = torch.tensor([feats_aligned], dtype=torch.float32).to(device)

            out = model(**enc, features=feat_tensor)
            logits = out["logits"] if isinstance(out, dict) else out.logits
            preds = logits.argmax(-1)[0].cpu().tolist()

            word_preds, seen = [], None
            for p, w in zip(preds, word_ids):
                if w is None or w == seen:
                    continue
                word_preds.append(p); seen = w

            if 1 not in word_preds:
                pending.extend(window)
                idx += seq_len
                continue

            for j, (tok_word, lab) in enumerate(zip(window, word_preds)):
                pending.append(tok_word)
                if lab == 1:
                    sentences.append(" ".join(pending))
                    pending = []
                    idx += j + 1
                    break

        if pending:
            sentences.append(" ".join(pending))

    return sentences

# Model

In [109]:
class UmBERToForSentenceSplit(PreTrainedModel):
    config_class = None

    def __init__(self, base_model_name: str, num_labels: int = 2, feat_dim: int = 2, dropout: float = DROPOUT):
        from transformers import AutoConfig

        cfg = AutoConfig.from_pretrained(base_model_name)
        super().__init__(cfg)
        self.umberto     = AutoModel.from_pretrained(base_model_name, add_pooling_layer=False, config=cfg)
        self.dropout     = nn.Dropout(dropout)
        self.feat_proj   = nn.Linear(feat_dim, cfg.hidden_size)
        self.classifier  = nn.Linear(cfg.hidden_size * 2, num_labels)
        self.num_labels  = num_labels

    def forward(self, input_ids=None, attention_mask=None, features=None, labels=None):
        out = self.umberto(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
        h   = self.dropout(out.last_hidden_state)              # (B, T, H)
        feat_h = torch.tanh(self.feat_proj(features.float()))  # (B, T, H)
        concat = torch.cat([h, feat_h], dim=-1)                # (B, T, 2H)
        logits = self.classifier(concat)                       # (B, T, C)

        loss = None
        if labels is not None:
            ce = nn.CrossEntropyLoss(ignore_index=-100)
            loss = ce(logits.view(-1, self.num_labels), labels.view(-1))
        return {"loss": loss, "logits": logits}

In [110]:
class FocalTrainer(Trainer):
    def __init__(self, class_weights: torch.Tensor, gamma: float, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.gamma = gamma

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels   = inputs.pop("labels")
        features = inputs.pop("features")
        outputs  = model(**inputs, features=features, labels=None)
        logits   = outputs["logits"]

        ce = nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device),
            ignore_index=-100,
            reduction="none",
        )
        ce_loss = ce(logits.view(-1, logits.size(-1)), labels.view(-1))
        pt      = torch.exp(-ce_loss)
        focal   = ((1 - pt) ** self.gamma) * ce_loss
        loss    = focal.mean()
        return (loss, outputs) if return_outputs else loss


In [111]:
LABEL_PAD_ID = -100

@dataclass
class DataCollatorWithFeatures:
    tokenizer: PreTrainedTokenizerBase
    label_pad_token_id: int = LABEL_PAD_ID

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        labels_list   = [f.pop("labels")   for f in features]
        feats_list    = [f.pop("features") for f in features]

        batch = self.tokenizer.pad(
            features,
            padding=True,
            return_tensors="pt",
        )

        max_len = batch["input_ids"].shape[1]

        padded_labels = []
        for lab in labels_list:
            pad = max_len - len(lab)
            if pad > 0:
                lab = lab + [self.label_pad_token_id] * pad
            padded_labels.append(lab)
        batch["labels"] = torch.tensor(padded_labels, dtype=torch.long)

        padded_feats = []
        for feats in feats_list:
            pad = max_len - len(feats)
            if pad > 0:
                feats = feats + [[0, 0]] * pad
            padded_feats.append(feats)
        batch["features"] = torch.tensor(padded_feats, dtype=torch.float32)

        return batch

# Main

In [112]:
set_seed(1)
wandb.init(project=PROJECT, name=RUN_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorWithFeatures(tokenizer)

In [113]:
# 1 Data & class‑weights
ds = load_dataset()
cls_w = torch.tensor(compute_class_weights(ds["train"]["labels"]), dtype=torch.float)

In [114]:
# 2 Tokenisation
ds_tok = ds.map(tokenize_and_align, batched=True, remove_columns=["tokens", "labels"], num_proc=4)

Map (num_proc=4):   0%|          | 0/1169 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/146 [00:00<?, ? examples/s]

In [115]:
# 3 Model
model = UmBERToForSentenceSplit(MODEL_NAME)
# freeze embeddings + first N layers
for p in model.umberto.embeddings.parameters():
    p.requires_grad = False
for layer in model.umberto.encoder.layer[:FREEZE_N]:
    for p in layer.parameters():
        p.requires_grad = False

In [116]:
# 4 Training args
args = TrainingArguments(
    output_dir="checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    fp16=True,
    run_name=RUN_NAME,
    report_to=["wandb"],
    seed=1,
)

In [117]:
# 5 Trainer
trainer = FocalTrainer(
    class_weights=cls_w,
    gamma=FOCAL_GAMMA,
    model=model,
    args=args,
    tokenizer=tokenizer,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["dev"],
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

/tmp/ipython-input-4279769487.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `FocalTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)


In [118]:
trainer.train()

You're using a CamembertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss,Accuracy,Precision Eos,Recall Eos,F1 Eos
1,0.010100,0.003203,0.996789,0.915254,1.000000,0.955752
2,0.002500,0.002421,0.998502,0.961310,0.996914,0.978788
3,0.001900,0.003194,0.998716,0.967066,0.996914,0.981763
4,0.001700,0.002374,0.998930,0.972892,0.996914,0.984756
5,0.001100,0.001132,0.999144,0.975904,1.000000,0.987805
6,0.001600,0.001231,0.998716,0.964286,1.000000,0.981818
7,0.000600,0.001849,0.999251,0.981763,0.996914,0.989280
8,0.000800,0.001801,0.999358,0.981818,1.000000,0.990826
9,0.000500,0.002761,0.999144,0.981707,0.993827,0.987730
10,0.002300,0.001844,0.999251,0.981763,0.996914,0.989280


TrainOutput(global_step=1480, training_loss=0.00359025242120166, metrics={'train_runtime': 336.6695, 'train_samples_per_second': 69.445, 'train_steps_per_second': 4.396, 'total_flos': 1073696940802092.0, 'train_loss': 0.00359025242120166, 'epoch': 20.0})

In [119]:
# 6 DEV DIAGNOSTICS – classification report & confusion matrix
dev_eval = trainer.evaluate()
print("\nDev metrics:", dev_eval)

preds_out = trainer.predict(ds_tok["dev"])
y_true = preds_out.label_ids.ravel()
y_pred = preds_out.predictions.argmax(-1).ravel()
mask = y_true != -100
y_true, y_pred = y_true[mask], y_pred[mask]


Dev metrics: {'eval_loss': 0.002449777675792575, 'eval_accuracy': 0.9992507759820186, 'eval_precision_EOS': 0.9817629179331308, 'eval_recall_EOS': 0.9969135802469136, 'eval_f1_EOS': 0.9892802450229708, 'eval_runtime': 0.3387, 'eval_samples_per_second': 431.014, 'eval_steps_per_second': 14.761, 'epoch': 20.0}


In [120]:
# classification report (printed & optionally logged as text/html)
clf_rep = classification_report(y_true, y_pred, target_names=["NOT_EOS", "EOS"])
print("\nClassification report (Dev):\n", clf_rep)


Classification report (Dev):
               precision    recall  f1-score   support

     NOT_EOS       1.00      1.00      1.00      9019
         EOS       0.98      1.00      0.99       324

    accuracy                           1.00      9343
   macro avg       0.99      1.00      0.99      9343
weighted avg       1.00      1.00      1.00      9343



In [121]:
# confusion matrix plot
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["NOT_EOS", "EOS"],
    yticklabels=["NOT_EOS", "EOS"],
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix – Dev")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FILES,"confusion_matrix_dev.png"))
plt.close()
wandb.log({"confusion_matrix": wandb.Image(os.path.join(OUTPUT_FILES,"confusion_matrix_dev.png"))})

In [122]:
# 8 | Save artefacts
trainer.save_model(OUTPUT_PATH)
tokenizer.save_pretrained(OUTPUT_PATH)

('/content/drive/MyDrive/AIRO/MNLP/homework 2/Task 1 - model 2/models/tokenizer_config.json',
 '/content/drive/MyDrive/AIRO/MNLP/homework 2/Task 1 - model 2/models/special_tokens_map.json',
 '/content/drive/MyDrive/AIRO/MNLP/homework 2/Task 1 - model 2/models/sentencepiece.bpe.model',
 '/content/drive/MyDrive/AIRO/MNLP/homework 2/Task 1 - model 2/models/added_tokens.json',
 '/content/drive/MyDrive/AIRO/MNLP/homework 2/Task 1 - model 2/models/tokenizer.json')

# Inference

In [123]:
tokens_flat, labels_flat = read_tokens(TEST_CSV)
chunks_toks, chunks_labs = chunkify(tokens_flat, labels_flat, SEQ_LEN)
test_ds = Dataset.from_dict({"tokens": chunks_toks, "labels": chunks_labs})

try:
    enc_test = test_ds.map(tokenize_and_align, batched=True, batch_size=128)
except Exception as e:
    print("Uso 'tokenize' standard:", e)
    enc_test = test_ds.map(tokenize, batched=True, batch_size=128)

Map:   0%|          | 0/24 [00:00<?, ? examples/s]

In [124]:
args = SimpleNamespace(per_device_eval_batch_size=BATCH_SIZE)
eval_trainer = trainer
pred_out = eval_trainer.predict(enc_test)
metrics = pred_out.metrics
print("Metriche (token‑level)")
for k,v in metrics.items():
    try:
        print(f"{k}: {float(v):.4f}")
    except Exception:
        print(k, v)

raw_preds = pred_out.predictions
if isinstance(raw_preds, tuple):
    raw_preds = raw_preds[0]
pred_ids = raw_preds.argmax(-1)

y_true, y_pred = [], []
for lab_row, pred_row in zip(enc_test["labels"], pred_ids):
    for y, yhat in zip(lab_row, pred_row):
        if y != -100:
            y_true.append(int(y))
            y_pred.append(int(yhat))

Metriche (token‑level)
test_loss: 0.1508
test_accuracy: 0.9855
test_precision_EOS: 0.8854
test_recall_EOS: 0.8854
test_f1_EOS: 0.8854
test_runtime: 0.0931
test_samples_per_second: 257.8160
test_steps_per_second: 10.7420


In [125]:
print("Classification report (Test)")
print(classification_report(y_true, y_pred, zero_division=0, target_names=["NOT_EOS", "EOS"]))

Classification report (Test)
              precision    recall  f1-score   support

     NOT_EOS       0.99      0.99      0.99      1426
         EOS       0.89      0.89      0.89        96

    accuracy                           0.99      1522
   macro avg       0.94      0.94      0.94      1522
weighted avg       0.99      0.99      0.99      1522



In [126]:
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=["NOT_EOS", "EOS"],
    yticklabels=["NOT_EOS", "EOS"],
)

plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix – Test")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FILES,"confusion_matrix_test.png"))
plt.close()
wandb.log({"confusion_matrix": wandb.Image(os.path.join(OUTPUT_FILES,"confusion_matrix_test.png"))})

In [127]:
input = []
for col in enc_test["tokens"]:
  for word in col:
    input.append(word) if word != "[PAD]" else None

In [128]:
rows = {"token":input, "label_predicted":y_pred}

df_out = pd.DataFrame(rows)
df_out.to_csv(csv_path, index=False)
print("Salvato:", csv_path)


wandb.finish()

Salvato: /content/drive/MyDrive/AIRO/MNLP/homework 2/Task 1 - model 2/output/test_predictions.csv


eval/accuracy,▁▆▆▇▇▆██▇██▇▇▇███████
eval/f1_EOS,▁▆▆▇▇▆██▇██▇▇▇███████
eval/loss,▅▄▅▄▁▁▃▂▄▃▂▅█▅▂▃▄▄▄▄▄
eval/precision_EOS,▁▆▆▇▇▆███████████████
eval/recall_EOS,█▄▄▄██▄█▁▄█▁▁▁█▄▄▄▄▄▄
eval/runtime,▁▂▁▃▂▂▆▁▅▂▂▅▄▂▂▂▆▆▂▂█
eval/samples_per_second,█▇█▆▆▆▂▇▃▇▇▄▄▇▇▇▃▃▇▆▁
eval/steps_per_second,█▇█▆▆▆▂▇▃▇▇▄▄▇▇▇▃▃▇▆▁
test/accuracy,█▁
test/f1_EOS,█▁
+11,...
